In [1]:
import pandas as pd
import numpy as np

print("="*60)
print("CREATING MASTER CSV FILE FOR POWER BI")
print("="*60)

# Load original data
print("\n📂 Loading original data...")
df = pd.read_csv('online_retail.csv', encoding='latin1')
print(f"   Loaded {len(df):,} rows")

# ============================================
# DATA CLEANING
# ============================================
print("\n🧹 Cleaning data...")

# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])
print(f"   Removed rows with missing CustomerID")

# Remove cancelled orders (InvoiceNo starting with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"   Removed cancelled orders")

# Remove negative quantities and prices
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
print(f"   Removed negative quantities and prices")

# Convert CustomerID to integer
df['CustomerID'] = df['CustomerID'].astype(int)

# Remove rows with missing Description
df = df.dropna(subset=['Description'])

print(f"✅ Cleaned data: {len(df):,} rows")

# ============================================
# CREATE NEW COLUMNS FOR ANALYSIS
# ============================================
print("\n📊 Creating analysis columns...")

# Revenue
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Date components
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Month_Name'] = df['InvoiceDate'].dt.strftime('%B')
df['Month_Year'] = df['InvoiceDate'].dt.strftime('%Y-%m')
df['Day'] = df['InvoiceDate'].dt.day
df['Day_Of_Week'] = df['InvoiceDate'].dt.dayofweek
df['Weekday_Name'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour
df['Date_Only'] = df['InvoiceDate'].dt.date

# Remove POSTAGE and non-product entries
non_products = ['POSTAGE', 'Manual', 'Bank Charges', 'Discount', 'DOTCOM POSTAGE']
df = df[~df['Description'].str.upper().isin(non_products)]

# Product categories (from StockCode prefix)
df['Category'] = df['StockCode'].astype(str).str[:2]

# Clean Country names
country_fixes = {
    'EIRE': 'Ireland',
    'USA': 'United States',
    'RSA': 'South Africa',
    'Channel Islands': 'United Kingdom'
}
df['Country'] = df['Country'].replace(country_fixes)

# ============================================
# REMOVE OUTLIERS (Optional)
# ============================================
print("\n📊 Removing statistical outliers...")

# Remove top 1% of orders by quantity
qty_99th = df['Quantity'].quantile(0.99)
before = len(df)
df = df[df['Quantity'] <= qty_99th]
print(f"   Removed {before - len(df):,} rows with quantity above 99th percentile")

# Remove top 1% of orders by price
price_99th = df['UnitPrice'].quantile(0.99)
before = len(df)
df = df[df['UnitPrice'] <= price_99th]
print(f"   Removed {before - len(df):,} rows with price above 99th percentile")

# ============================================
# CREATE MASTER CSV
# ============================================
print("\n💾 Creating master CSV file...")

# Select columns in logical order
master_columns = [
    'InvoiceNo', 'InvoiceDate', 'Year', 'Month', 'Month_Name', 'Month_Year',
    'Day', 'Day_Of_Week', 'Weekday_Name', 'Hour', 'Date_Only',
    'CustomerID', 'Country', 'StockCode', 'Description', 'Category',
    'Quantity', 'UnitPrice', 'Revenue'
]

df_master = df[master_columns]

# Sort by date
df_master = df_master.sort_values('InvoiceDate')

# Save to CSV
df_master.to_csv('online_retail_master.csv', index=False)

print(f"✅ Master CSV saved: online_retail_master.csv")
print(f"   Rows: {len(df_master):,}")
print(f"   Columns: {len(df_master.columns)}")

# ============================================
# SHOW PREVIEW
# ============================================
print("\n📋 Master CSV Preview:")
print(df_master.head(10).to_string())

# ============================================
# CREATE SUMMARY STATISTICS
# ============================================
print("\n" + "="*60)
print("📊 SUMMARY STATISTICS")
print("="*60)

total_revenue = df_master['Revenue'].sum()
total_orders = df_master['InvoiceNo'].nunique()
total_customers = df_master['CustomerID'].nunique()
avg_order_value = total_revenue / total_orders

print(f"💰 Total Revenue:      £{total_revenue:,.2f}")
print(f"📦 Total Orders:       {total_orders:,}")
print(f"👥 Total Customers:    {total_customers:,}")
print(f"💳 Avg Order Value:    £{avg_order_value:.2f}")
print(f"🌍 Unique Countries:   {df_master['Country'].nunique()}")
print(f"🏷️ Unique Products:    {df_master['Description'].nunique()}")
print(f"📅 Date Range:         {df_master['InvoiceDate'].min().date()} to {df_master['InvoiceDate'].max().date()}")

print("\n✅ READY FOR POWER BI!")
print("   Import 'online_retail_master.csv' into Power BI Desktop")

CREATING MASTER CSV FILE FOR POWER BI

📂 Loading original data...
   Loaded 541,909 rows

🧹 Cleaning data...
   Removed rows with missing CustomerID
   Removed cancelled orders
   Removed negative quantities and prices
✅ Cleaned data: 397,884 rows

📊 Creating analysis columns...

📊 Removing statistical outliers...
   Removed 3,890 rows with quantity above 99th percentile
   Removed 3,733 rows with price above 99th percentile

💾 Creating master CSV file...
✅ Master CSV saved: online_retail_master.csv
   Rows: 389,146
   Columns: 19

📋 Master CSV Preview:
   InvoiceNo         InvoiceDate  Year  Month Month_Name Month_Year  Day  Day_Of_Week Weekday_Name  Hour   Date_Only  CustomerID         Country StockCode                          Description Category  Quantity  UnitPrice  Revenue
0     536365 2010-12-01 08:26:00  2010     12   December    2010-12    1            2    Wednesday     8  2010-12-01       17850  United Kingdom    85123A   WHITE HANGING HEART T-LIGHT HOLDER       85         

In [4]:
import pandas as pd
import numpy as np

print("="*60)
print("COMBINING MULTIPLE CSV FILES INTO ONE MASTER FILE")
print("="*60)

# ============================================
# LOAD YOUR EXISTING CSV FILES
# ============================================

print("\n📂 Loading your existing CSV files...")

# Load each file
monthly = pd.read_csv('monthly_revenue.csv')
top_products = pd.read_csv('top_products.csv')
country = pd.read_csv('country_revenue.csv')
hourly = pd.read_csv('hourly_sales.csv')
kpi = pd.read_csv('kpi_wide.csv')

print(f"✅ monthly_revenue.csv: {len(monthly)} rows")
print(f"✅ top_products.csv: {len(top_products)} rows")
print(f"✅ country_revenue.csv: {len(country)} rows")
print(f"✅ hourly_sales.csv: {len(hourly)} rows")
print(f"✅ kpi_wide.csv: {len(kpi)} rows")

# ============================================
# DISPLAY WHAT EACH FILE CONTAINS
# ============================================

print("\n" + "="*60)
print("📋 FILE STRUCTURES:")
print("="*60)

print("\n📊 monthly_revenue.csv columns:")
print(monthly.columns.tolist())
print(monthly.head(3))

print("\n🏆 top_products.csv columns:")
print(top_products.columns.tolist())
print(top_products.head(3))

print("\n🌍 country_revenue.csv columns:")
print(country.columns.tolist())
print(country.head(3))

print("\n⏰ hourly_sales.csv columns:")
print(hourly.columns.tolist())
print(hourly.head(3))

print("\n💳 kpi_wide.csv columns:")
print(kpi.columns.tolist())
print(kpi.head(3))

# ============================================
# CREATE MASTER FILE STRUCTURE
# ============================================

print("\n" + "="*60)
print("🔧 CREATING MASTER FILE...")
print("="*60)

# Since your CSV files have different structures (monthly, products, countries, hourly),
# we need to create a MASTER file that has ALL columns for Power BI filtering

# Create a master dataframe with key columns
master_data = []

# Add monthly revenue data with 'Month' as identifier
for _, row in monthly.iterrows():
    master_data.append({
        'Report_Type': 'Monthly Revenue',
        'Dimension': row['Month'],
        'Revenue': row['Revenue'],
        'Category': 'Trend Analysis'
    })

# Add top products data
for _, row in top_products.iterrows():
    master_data.append({
        'Report_Type': 'Top Products',
        'Dimension': row['Product'],
        'Revenue': row['Revenue'],
        'Category': 'Product Analysis'
    })

# Add country revenue data
for _, row in country.iterrows():
    master_data.append({
        'Report_Type': 'Country Revenue',
        'Dimension': row['Country'],
        'Revenue': row['Revenue'],
        'Category': 'Geographic Analysis'
    })

# Add hourly sales data
for _, row in hourly.iterrows():
    master_data.append({
        'Report_Type': 'Hourly Sales',
        'Dimension': str(row['Hour']),
        'Revenue': row['Revenue'],
        'Category': 'Time Analysis'
    })

# Create master dataframe
master_df = pd.DataFrame(master_data)

print(f"✅ Master file created with {len(master_df)} rows")

# ============================================
# ADD KPI DATA AS SEPARATE FILE
# ============================================

print("\n💾 Saving files...")

# Save master file
master_df.to_csv('powerbi_master_data.csv', index=False)
print(f"✅ powerbi_master_data.csv - {len(master_df)} rows")

# Save KPI file separately (for scorecards)
kpi.to_csv('powerbi_kpi_data.csv', index=False)
print(f"✅ powerbi_kpi_data.csv - {len(kpi)} rows")

# ============================================
# CREATE A BETTER STRUCTURE: LONG FORMAT FOR POWER BI
# ============================================

print("\n" + "="*60)
print("📊 CREATING POWER-BI READY FORMAT...")
print("="*60)

# Each chart needs its own table for best performance
# But they will share a common Country filter

# 1. Monthly revenue with date formatting
monthly_powerbi = monthly.copy()
monthly_powerbi['Report_Type'] = 'Monthly'
monthly_powerbi['Chart_Type'] = 'Line Chart'

# 2. Top products with ranking
top_products_powerbi = top_products.copy()
top_products_powerbi['Rank'] = range(1, len(top_products_powerbi) + 1)
top_products_powerbi['Report_Type'] = 'Products'

# 3. Country revenue with percentage
total_revenue = country['Revenue'].sum()
country_powerbi = country.copy()
country_powerbi['Percentage'] = (country_powerbi['Revenue'] / total_revenue) * 100
country_powerbi['Report_Type'] = 'Countries'

# 4. Hourly sales with hour formatting
hourly_powerbi = hourly.copy()
hourly_powerbi['Hour_Label'] = hourly_powerbi['Hour'].astype(str) + ':00'
hourly_powerbi['Report_Type'] = 'Hourly'

# ============================================
# SAVE ALL FILES
# ============================================

print("\n💾 Saving Power-BI ready files...")

monthly_powerbi.to_csv('powerbi_monthly.csv', index=False)
top_products_powerbi.to_csv('powerbi_products.csv', index=False)
country_powerbi.to_csv('powerbi_countries.csv', index=False)
hourly_powerbi.to_csv('powerbi_hourly.csv', index=False)

print("✅ powerbi_monthly.csv")
print("✅ powerbi_products.csv")
print("✅ powerbi_countries.csv")
print("✅ powerbi_hourly.csv")
print("✅ powerbi_kpi_data.csv (already saved)")

# ============================================
# VERIFICATION
# ============================================

print("\n" + "="*60)
print("📋 FILE SUMMARY FOR POWER BI")
print("=="*60)

print("\n📁 You now have these files ready for Power BI:")
print("   1. powerbi_kpi_data.csv - For KPI scorecards")
print("   2. powerbi_monthly.csv - For monthly trend chart")
print("   3. powerbi_products.csv - For top 10 products")
print("   4. powerbi_countries.csv - For country chart + slicer")
print("   5. powerbi_hourly.csv - For hourly sales chart")

print("\n🔗 TO MAKE SLICERS WORK ACROSS ALL CHARTS:")
print("   Import powerbi_countries.csv first for the Country slicer")
print("   Then import other files and create relationships using:")
print("   - For monthly: No direct relationship (uses date filter)")
print("   - For products: No Country filter needed")
print("   - For hourly: No Country filter needed")

print("\n✅ All files ready! Transfer to Windows for Power BI.")

COMBINING MULTIPLE CSV FILES INTO ONE MASTER FILE

📂 Loading your existing CSV files...
✅ monthly_revenue.csv: 13 rows
✅ top_products.csv: 10 rows
✅ country_revenue.csv: 10 rows
✅ hourly_sales.csv: 15 rows
✅ kpi_wide.csv: 1 rows

📋 FILE STRUCTURES:

📊 monthly_revenue.csv columns:
['MonthYear', 'Revenue']
  MonthYear    Revenue
0   2010-12  561966.50
1   2011-01  560679.74
2   2011-02  440878.62

🏆 top_products.csv columns:
['Description', 'Revenue']
                          Description    Revenue
0         PAPER CRAFT , LITTLE BIRDIE  168469.60
1            REGENCY CAKESTAND 3 TIER  141946.00
2  WHITE HANGING HEART T-LIGHT HOLDER  100046.95

🌍 country_revenue.csv columns:
['Country', 'Revenue']
          Country      Revenue
0  United Kingdom  7209013.343
1     Netherlands   283889.340
2            EIRE   256996.620

⏰ hourly_sales.csv columns:
['Hour', 'Revenue']
   Hour    Revenue
0     6       4.25
1     7   30469.21
2     8  277454.19

💳 kpi_wide.csv columns:
['Total_Revenue', 'To

KeyError: 'Month'

In [5]:
import pandas as pd

print("="*60)
print("CHECKING YOUR CSV FILE STRUCTURES")
print("="*60)

# Load and display column names for each file
monthly = pd.read_csv('monthly_revenue.csv')
print("\n📊 monthly_revenue.csv columns:")
print(monthly.columns.tolist())
print("First few rows:")
print(monthly.head(3))

top_products = pd.read_csv('top_products.csv')
print("\n🏆 top_products.csv columns:")
print(top_products.columns.tolist())
print("First few rows:")
print(top_products.head(3))

country = pd.read_csv('country_revenue.csv')
print("\n🌍 country_revenue.csv columns:")
print(country.columns.tolist())
print("First few rows:")
print(country.head(3))

hourly = pd.read_csv('hourly_sales.csv')
print("\n⏰ hourly_sales.csv columns:")
print(hourly.columns.tolist())
print("First few rows:")
print(hourly.head(3))

kpi = pd.read_csv('kpi_wide.csv')
print("\n💳 kpi_wide.csv columns:")
print(kpi.columns.tolist())
print("First few rows:")
print(kpi.head(3))


CHECKING YOUR CSV FILE STRUCTURES

📊 monthly_revenue.csv columns:
['MonthYear', 'Revenue']
First few rows:
  MonthYear    Revenue
0   2010-12  561966.50
1   2011-01  560679.74
2   2011-02  440878.62

🏆 top_products.csv columns:
['Description', 'Revenue']
First few rows:
                          Description    Revenue
0         PAPER CRAFT , LITTLE BIRDIE  168469.60
1            REGENCY CAKESTAND 3 TIER  141946.00
2  WHITE HANGING HEART T-LIGHT HOLDER  100046.95

🌍 country_revenue.csv columns:
['Country', 'Revenue']
First few rows:
          Country      Revenue
0  United Kingdom  7209013.343
1     Netherlands   283889.340
2            EIRE   256996.620

⏰ hourly_sales.csv columns:
['Hour', 'Revenue']
First few rows:
   Hour    Revenue
0     6       4.25
1     7   30469.21
2     8  277454.19

💳 kpi_wide.csv columns:
['Total_Revenue', 'Total_Orders', 'Total_Customers', 'Avg_Order_Value']
First few rows:
   Total_Revenue  Total_Orders  Total_Customers  Avg_Order_Value
0    8702521.753   

In [6]:
import pandas as pd
import numpy as np

print("="*60)
print("COMBINING MULTIPLE CSV FILES INTO ONE MASTER FILE")
print("="*60)

# ============================================
# LOAD YOUR EXISTING CSV FILES
# ============================================

print("\n📂 Loading your existing CSV files...")

# Load each file (using your actual column names)
monthly = pd.read_csv('monthly_revenue.csv')
top_products = pd.read_csv('top_products.csv')
country = pd.read_csv('country_revenue.csv')
hourly = pd.read_csv('hourly_sales.csv')
kpi = pd.read_csv('kpi_wide.csv')

print(f"✅ monthly_revenue.csv: {len(monthly)} rows")
print(f"✅ top_products.csv: {len(top_products)} rows")
print(f"✅ country_revenue.csv: {len(country)} rows")
print(f"✅ hourly_sales.csv: {len(hourly)} rows")
print(f"✅ kpi_wide.csv: 1 row with {len(kpi.columns)} KPIs")

# ============================================
# CREATE MASTER FILE (Combined for reference)
# ============================================

print("\n" + "="*60)
print("🔧 CREATING MASTER FILE...")
print("="*60)

# Create a master dataframe with all data combined
master_data = []

# Add monthly revenue data
for _, row in monthly.iterrows():
    master_data.append({
        'Report_Type': 'Monthly Revenue',
        'Dimension': row['MonthYear'],
        'Revenue': row['Revenue'],
        'Category': 'Trend Analysis'
    })

# Add top products data
for _, row in top_products.iterrows():
    master_data.append({
        'Report_Type': 'Top Products',
        'Dimension': row['Description'],
        'Revenue': row['Revenue'],
        'Category': 'Product Analysis'
    })

# Add country revenue data
for _, row in country.iterrows():
    master_data.append({
        'Report_Type': 'Country Revenue',
        'Dimension': row['Country'],
        'Revenue': row['Revenue'],
        'Category': 'Geographic Analysis'
    })

# Add hourly sales data
for _, row in hourly.iterrows():
    master_data.append({
        'Report_Type': 'Hourly Sales',
        'Dimension': str(int(row['Hour'])) + ':00',
        'Revenue': row['Revenue'],
        'Category': 'Time Analysis'
    })

# Create master dataframe
master_df = pd.DataFrame(master_data)

print(f"✅ Master file created with {len(master_df)} rows")
print(f"\n📋 Master file preview:")
print(master_df.head(10))

# Save master file
master_df.to_csv('powerbi_master_data.csv', index=False)
print(f"\n✅ powerbi_master_data.csv saved")

# ============================================
# SAVE INDIVIDUAL FILES FOR POWER BI (Standardized)
# ============================================

print("\n" + "="*60)
print("💾 SAVING POWER BI READY FILES")
print("="*60)

# 1. Monthly Revenue - keep as is
monthly.to_csv('powerbi_monthly.csv', index=False)
print("✅ powerbi_monthly.csv - Columns: MonthYear, Revenue")

# 2. Top Products - keep as is
top_products.to_csv('powerbi_products.csv', index=False)
print("✅ powerbi_products.csv - Columns: Description, Revenue")

# 3. Country Revenue - keep as is
country.to_csv('powerbi_countries.csv', index=False)
print("✅ powerbi_countries.csv - Columns: Country, Revenue")

# 4. Hourly Sales - keep as is
hourly.to_csv('powerbi_hourly.csv', index=False)
print("✅ powerbi_hourly.csv - Columns: Hour, Revenue")

# 5. KPI data - keep as is
kpi.to_csv('powerbi_kpi_data.csv', index=False)
print("✅ powerbi_kpi_data.csv - Columns: Total_Revenue, Total_Orders, Total_Customers, Avg_Order_Value")

# ============================================
# CREATE ADDITIONAL USEFUL FILES
# ============================================

print("\n" + "="*60)
print("📊 CREATING ADDITIONAL USEFUL FILES")
print("="*60)

# 6. Create a Calendar/Date table for better time filtering
calendar_data = []
for month in monthly['MonthYear'].unique():
    calendar_data.append({
        'MonthYear': month,
        'Year': int(month[:4]),
        'Month': int(month[5:7]),
        'Month_Name': pd.to_datetime(month).strftime('%B')
    })
calendar_df = pd.DataFrame(calendar_data)
calendar_df.to_csv('powerbi_calendar.csv', index=False)
print("✅ powerbi_calendar.csv - For date filtering")

# 7. Create a summary file for quick reference
summary_data = {
    'Metric': ['Total Revenue', 'Total Orders', 'Total Customers', 'Avg Order Value'],
    'Value': [
        kpi['Total_Revenue'].iloc[0],
        kpi['Total_Orders'].iloc[0],
        kpi['Total_Customers'].iloc[0],
        kpi['Avg_Order_Value'].iloc[0]
    ]
}
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('powerbi_summary.csv', index=False)
print("✅ powerbi_summary.csv - Quick reference")

# ============================================
# VERIFICATION
# ============================================

print("\n" + "="*60)
print("✅ ALL FILES CREATED SUCCESSFULLY!")
print("="*60)

print("\n📁 Files ready for Power BI (transfer these to Windows):")
print("-"*50)
print("   1. powerbi_kpi_data.csv     ← For 4 KPI scorecards")
print("   2. powerbi_monthly.csv      ← Monthly revenue line chart")
print("   3. powerbi_products.csv     ← Top 10 products bar chart")
print("   4. powerbi_countries.csv    ← Country revenue + Country slicer")
print("   5. powerbi_hourly.csv       ← Hourly sales line chart")
print("   6. powerbi_calendar.csv     ← Date filtering (optional)")
print("   7. powerbi_summary.csv      ← Quick reference")
print("   8. powerbi_master_data.csv  ← Combined data (optional)")

print("\n📊 Sample of powerbi_monthly.csv:")
print(monthly.head(5))

print("\n📊 Sample of powerbi_countries.csv:")
print(country.head(5))

print("\n" + "="*60)
print("🎯 NEXT STEPS:")
print("="*60)
print("1. Copy these CSV files to your Windows PC")
print("2. Open Power BI Desktop")
print("3. Import each file (Get Data → Text/CSV)")
print("4. Build your dashboard using the guides I provided")
print("\n✅ Ready for transfer!")

COMBINING MULTIPLE CSV FILES INTO ONE MASTER FILE

📂 Loading your existing CSV files...
✅ monthly_revenue.csv: 13 rows
✅ top_products.csv: 10 rows
✅ country_revenue.csv: 10 rows
✅ hourly_sales.csv: 15 rows
✅ kpi_wide.csv: 1 row with 4 KPIs

🔧 CREATING MASTER FILE...
✅ Master file created with 48 rows

📋 Master file preview:
       Report_Type Dimension     Revenue        Category
0  Monthly Revenue   2010-12  561966.500  Trend Analysis
1  Monthly Revenue   2011-01  560679.740  Trend Analysis
2  Monthly Revenue   2011-02  440878.620  Trend Analysis
3  Monthly Revenue   2011-03  581693.740  Trend Analysis
4  Monthly Revenue   2011-04  453462.371  Trend Analysis
5  Monthly Revenue   2011-05  657825.940  Trend Analysis
6  Monthly Revenue   2011-06  650141.410  Trend Analysis
7  Monthly Revenue   2011-07  590342.251  Trend Analysis
8  Monthly Revenue   2011-08  633889.240  Trend Analysis
9  Monthly Revenue   2011-09  935926.361  Trend Analysis

✅ powerbi_master_data.csv saved

💾 SAVING POWE

In [7]:
import pandas as pd
import numpy as np

print("="*70)
print("CREATING ERROR-FREE MASTER CSV FOR POWER BI")
print("="*70)

# ============================================
# STEP 1: LOAD ORIGINAL DATA
# ============================================

print("\n📂 STEP 1: Loading original data...")
df = pd.read_csv('online_retail.csv', encoding='latin1')
print(f"   ✅ Loaded {len(df):,} rows and {len(df.columns)} columns")

# ============================================
# STEP 2: BASIC DATA CLEANING
# ============================================

print("\n🧹 STEP 2: Cleaning data...")

# Remove missing CustomerID
before = len(df)
df = df.dropna(subset=['CustomerID'])
print(f"   Removed {before - len(df):,} rows with missing CustomerID")

# Remove cancelled orders (InvoiceNo starting with 'C')
before = len(df)
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"   Removed {before - len(df):,} cancelled orders")

# Remove negative quantities
before = len(df)
df = df[df['Quantity'] > 0]
print(f"   Removed {before - len(df):,} rows with negative quantities")

# Remove negative/zero prices
before = len(df)
df = df[df['UnitPrice'] > 0]
print(f"   Removed {before - len(df):,} rows with invalid prices")

# Convert CustomerID to integer (remove .0)
df['CustomerID'] = df['CustomerID'].astype(int)

print(f"\n   ✅ Cleaned data: {len(df):,} rows")

# ============================================
# STEP 3: CREATE REVENUE AND DATE COLUMNS
# ============================================

print("\n📊 STEP 3: Creating calculated columns...")

# Revenue
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Convert to datetime (handle errors gracefully)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Remove rows with invalid dates
before = len(df)
df = df.dropna(subset=['InvoiceDate'])
print(f"   Removed {before - len(df):,} rows with invalid dates")

# Create date components
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['MonthYear'] = df['InvoiceDate'].dt.strftime('%Y-%m')
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek

print(f"   Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

# ============================================
# STEP 4: CLEAN TEXT COLUMNS
# ============================================

print("\n✏️ STEP 4: Cleaning text columns...")

# Remove special characters from Description
df['Description'] = df['Description'].astype(str).str.replace(r'[^\w\s]', '', regex=True)

# Clean Country names (standardize)
country_mapping = {
    'EIRE': 'Ireland',
    'USA': 'United States',
    'RSA': 'South Africa',
    'Channel Islands': 'United Kingdom',
    'Unspecified': 'Unknown'
}
df['Country'] = df['Country'].replace(country_mapping)

# Remove any remaining nulls in Country
df['Country'] = df['Country'].fillna('Unknown')

print(f"   Unique countries: {df['Country'].nunique()}")

# ============================================
# STEP 5: REMOVE OUTLIERS (Cleaner data)
# ============================================

print("\n📈 STEP 5: Removing statistical outliers...")

# Remove quantity outliers (top 1%)
qty_limit = df['Quantity'].quantile(0.99)
before = len(df)
df = df[df['Quantity'] <= qty_limit]
print(f"   Removed {before - len(df):,} rows with quantity > {qty_limit:.0f}")

# Remove price outliers (top 1%)
price_limit = df['UnitPrice'].quantile(0.99)
before = len(df)
df = df[df['UnitPrice'] <= price_limit]
print(f"   Removed {before - len(df):,} rows with price > £{price_limit:.2f}")

# ============================================
# STEP 6: CREATE MASTER CSV WITH CLEAN COLUMNS
# ============================================

print("\n💾 STEP 6: Creating master CSV file...")

# Select columns in logical order (all will be used in Power BI)
master_columns = [
    'InvoiceNo',
    'InvoiceDate', 
    'Year',
    'Month',
    'MonthYear',
    'Hour',
    'DayOfWeek',
    'CustomerID',
    'Country',
    'Description',
    'Quantity',
    'UnitPrice',
    'Revenue'
]

master_df = df[master_columns]

# Sort by date for better performance
master_df = master_df.sort_values('InvoiceDate')

# ============================================
# STEP 7: DATA TYPE VERIFICATION
# ============================================

print("\n🔍 STEP 7: Verifying data types...")

print("\n   Data types in master file:")
for col in master_df.columns:
    dtype = master_df[col].dtype
    print(f"   {col:<15} : {dtype}")

# ============================================
# STEP 8: CHECK FOR ANY NULL VALUES
# ============================================

print("\n🔍 STEP 8: Checking for null values...")
null_counts = master_df.isnull().sum()
null_columns = null_counts[null_counts > 0]

if len(null_columns) > 0:
    print("   ⚠️ Warning: Null values found:")
    for col, count in null_columns.items():
        print(f"      {col}: {count} nulls")
else:
    print("   ✅ No null values found in any column!")

# ============================================
# STEP 9: SAVE TO CSV
# ============================================

print("\n💾 STEP 9: Saving to CSV...")

# Save with proper settings for Power BI
master_df.to_csv('powerbi_master_clean.csv', 
                  index=False, 
                  encoding='utf-8',
                  float_format='%.2f')  # Forces 2 decimal places

print(f"\n   ✅ File saved: powerbi_master_clean.csv")
print(f"   📊 File size: {len(master_df):,} rows × {len(master_df.columns)} columns")
print(f"   💾 Location: Same folder as this notebook")

# ============================================
# STEP 10: CREATE SAMPLE PREVIEW
# ============================================

print("\n📋 STEP 10: Sample preview (first 10 rows):")
print("="*70)
print(master_df.head(10).to_string())

# ============================================
# STEP 11: CREATE KPIS FOR REFERENCE
# ============================================

print("\n📊 STEP 11: Key Performance Indicators")
print("="*70)

total_revenue = master_df['Revenue'].sum()
total_orders = master_df['InvoiceNo'].nunique()
total_customers = master_df['CustomerID'].nunique()
avg_order_value = total_revenue / total_orders

print(f"💰 Total Revenue:      £{total_revenue:,.2f}")
print(f"📦 Total Orders:       {total_orders:,}")
print(f"👥 Total Customers:    {total_customers:,}")
print(f"💳 Avg Order Value:    £{avg_order_value:.2f}")
print(f"🌍 Countries:          {master_df['Country'].nunique()}")
print(f"📅 Date range:         {master_df['InvoiceDate'].min().date()} to {master_df['InvoiceDate'].max().date()}")
print(f"🏷️ Unique Products:    {master_df['Description'].nunique()}")

# ============================================
# STEP 12: CREATE QUICK SUMMARY CSV (Optional)
# ============================================

print("\n💾 STEP 12: Creating summary CSV for quick reference...")

summary_df = pd.DataFrame({
    'Metric': ['Total Revenue', 'Total Orders', 'Total Customers', 'Average Order Value', 'Date Start', 'Date End'],
    'Value': [
        f'£{total_revenue:,.2f}',
        f'{total_orders:,}',
        f'{total_customers:,}',
        f'£{avg_order_value:.2f}',
        master_df['InvoiceDate'].min().date(),
        master_df['InvoiceDate'].max().date()
    ]
})
summary_df.to_csv('powerbi_summary_reference.csv', index=False)
print("   ✅ Saved: powerbi_summary_reference.csv")

# ============================================
# FINAL VERIFICATION
# ============================================

print("\n" + "="*70)
print("✅✅✅ MASTER CSV FILE CREATED SUCCESSFULLY! ✅✅✅")
print("="*70)

print("\n📁 FILES CREATED:")
print("   1. powerbi_master_clean.csv      ← MAIN FILE (use this in Power BI)")
print("   2. powerbi_summary_reference.csv ← Reference only")

print("\n🎯 WHAT MAKES THIS FILE ERROR-FREE:")
print("   ✅ No missing values in key columns")
print("   ✅ Cleaned country names (EIRE → Ireland)")
print("   ✅ Removed special characters from descriptions")
print("   ✅ Removed statistical outliers")
print("   ✅ Proper data types (numbers are numbers, text is text)")
print("   ✅ Dates in proper datetime format")
print("   ✅ 2 decimal places for all currency")
print("   ✅ UTF-8 encoding (no special character issues)")

print("\n📤 NEXT STEPS:")
print("   1. Copy 'powerbi_master_clean.csv' to your Windows PC")
print("   2. Open Power BI Desktop")
print("   3. Get Data → Text/CSV → Select this file")
print("   4. Click Load")
print("   5. Start building your dashboard!")

print("\n🚀 READY FOR POWER BI!")

CREATING ERROR-FREE MASTER CSV FOR POWER BI

📂 STEP 1: Loading original data...
   ✅ Loaded 541,909 rows and 8 columns

🧹 STEP 2: Cleaning data...
   Removed 135,080 rows with missing CustomerID
   Removed 8,905 cancelled orders
   Removed 0 rows with negative quantities
   Removed 40 rows with invalid prices

   ✅ Cleaned data: 397,884 rows

📊 STEP 3: Creating calculated columns...
   Removed 0 rows with invalid dates
   Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00

✏️ STEP 4: Cleaning text columns...
   Unique countries: 36

📈 STEP 5: Removing statistical outliers...
   Removed 3,890 rows with quantity > 120
   Removed 3,734 rows with price > £14.95

💾 STEP 6: Creating master CSV file...

🔍 STEP 7: Verifying data types...

   Data types in master file:
   InvoiceNo       : str
   InvoiceDate     : datetime64[us]
   Year            : int32
   Month           : int32
   MonthYear       : str
   Hour            : int32
   DayOfWeek       : int32
   CustomerID      : int64
   C

In [8]:
import pandas as pd
import numpy as np

print("="*70)
print("CREATING ERROR-FREE MASTER CSV FOR POWER BI")
print("="*70)

# ============================================
# STEP 1: LOAD ORIGINAL DATA
# ============================================

print("\n📂 STEP 1: Loading original data...")
df = pd.read_csv('online_retail.csv', encoding='latin1')
print(f"   ✅ Loaded {len(df):,} rows and {len(df.columns)} columns")

# ============================================
# STEP 2: BASIC DATA CLEANING
# ============================================

print("\n🧹 STEP 2: Cleaning data...")

# Remove missing CustomerID
before = len(df)
df = df.dropna(subset=['CustomerID'])
print(f"   Removed {before - len(df):,} rows with missing CustomerID")

# Remove cancelled orders (InvoiceNo starting with 'C')
before = len(df)
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"   Removed {before - len(df):,} cancelled orders")

# Remove negative quantities
before = len(df)
df = df[df['Quantity'] > 0]
print(f"   Removed {before - len(df):,} rows with negative quantities")

# Remove negative/zero prices
before = len(df)
df = df[df['UnitPrice'] > 0]
print(f"   Removed {before - len(df):,} rows with invalid prices")

# Convert CustomerID to integer (remove .0)
df['CustomerID'] = df['CustomerID'].astype(int)

print(f"\n   ✅ Cleaned data: {len(df):,} rows")

# ============================================
# STEP 3: REMOVE NON-PRODUCT ENTRIES (CRITICAL!)
# ============================================

print("\n🚫 STEP 3: Removing non-product entries...")

# List of non-product descriptions to remove
non_products = [
    'POSTAGE', 'Manual', 'Bank Charges', 'Discount', 'DOTCOM POSTAGE', 
    'C2', 'CARRIAGE', 'AMAZON', 'BANK CHARGES', 'PACKAGING', 
    'SHIPPING', 'DELIVERY', 'INSURANCE', 'GIFT WRAP', 'RETURNS'
]

# Remove rows with non-product descriptions
before = len(df)
df = df[~df['Description'].astype(str).str.upper().isin(non_products)]
print(f"   Removed {before - len(df):,} rows with non-product descriptions (POSTAGE, Manual, etc.)")

# Also remove rows with empty or null descriptions
before = len(df)
df = df.dropna(subset=['Description'])
df = df[df['Description'].astype(str).str.strip() != '']
print(f"   Removed {before - len(df):,} rows with empty descriptions")

# Remove rows with StockCode that indicates non-product (starting with 'M' or 'BANK')
before = len(df)
df = df[~df['StockCode'].astype(str).str.match(r'^M', na=False)]
print(f"   Removed {before - len(df):,} rows with 'M' stock codes (non-products)")

# Remove rows where Description contains non-product keywords
non_product_keywords = [
    'POSTAGE', 'MANUAL', 'BANK', 'DISCOUNT', 'DOTCOM', 'CARRIAGE',
    'SHIPPING', 'DELIVERY', 'INSURANCE', 'RETURN', 'GIFT'
]
before = len(df)
for keyword in non_product_keywords:
    df = df[~df['Description'].astype(str).str.upper().str.contains(keyword, na=False)]
print(f"   Removed rows containing non-product keywords")

print(f"\n   ✅ After removing non-products: {len(df):,} rows")

# ============================================
# STEP 4: CREATE REVENUE AND DATE COLUMNS
# ============================================

print("\n📊 STEP 4: Creating calculated columns...")

# Revenue
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Convert to datetime (handle errors gracefully)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

# Remove rows with invalid dates
before = len(df)
df = df.dropna(subset=['InvoiceDate'])
print(f"   Removed {before - len(df):,} rows with invalid dates")

# Create date components
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['MonthYear'] = df['InvoiceDate'].dt.strftime('%Y-%m')
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek

print(f"   Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

# ============================================
# STEP 5: CLEAN TEXT COLUMNS
# ============================================

print("\n✏️ STEP 5: Cleaning text columns...")

# Remove special characters from Description (but keep product names readable)
df['Description'] = df['Description'].astype(str).str.replace(r'[^\w\s-]', '', regex=True)
df['Description'] = df['Description'].str.strip()

# Clean Country names (standardize)
country_mapping = {
    'EIRE': 'Ireland',
    'USA': 'United States',
    'RSA': 'South Africa',
    'Channel Islands': 'United Kingdom',
    'Unspecified': 'Unknown'
}
df['Country'] = df['Country'].replace(country_mapping)

# Remove any remaining nulls in Country
df['Country'] = df['Country'].fillna('Unknown')

print(f"   Unique countries: {df['Country'].nunique()}")

# ============================================
# STEP 6: REMOVE OUTLIERS (Cleaner data)
# ============================================

print("\n📈 STEP 6: Removing statistical outliers...")

# Remove quantity outliers (top 1%)
qty_limit = df['Quantity'].quantile(0.99)
before = len(df)
df = df[df['Quantity'] <= qty_limit]
print(f"   Removed {before - len(df):,} rows with quantity > {qty_limit:.0f}")

# Remove price outliers (top 1%)
price_limit = df['UnitPrice'].quantile(0.99)
before = len(df)
df = df[df['UnitPrice'] <= price_limit]
print(f"   Removed {before - len(df):,} rows with price > £{price_limit:.2f}")

# ============================================
# STEP 7: VERIFY NO NON-PRODUCTS REMAIN
# ============================================

print("\n🔍 STEP 7: Verifying non-products removed...")

# Check for any remaining non-product keywords
remaining_issues = []
for keyword in non_product_keywords:
    count = df[df['Description'].astype(str).str.upper().str.contains(keyword, na=False)].shape[0]
    if count > 0:
        remaining_issues.append(f"{keyword}: {count} rows")

if len(remaining_issues) > 0:
    print("   ⚠️ Warning: Some non-products may remain:")
    for issue in remaining_issues:
        print(f"      {issue}")
else:
    print("   ✅ No non-product keywords found!")

# ============================================
# STEP 8: CREATE MASTER CSV WITH CLEAN COLUMNS
# ============================================

print("\n💾 STEP 8: Creating master CSV file...")

# Select columns in logical order (all will be used in Power BI)
master_columns = [
    'InvoiceNo',
    'InvoiceDate', 
    'Year',
    'Month',
    'MonthYear',
    'Hour',
    'DayOfWeek',
    'CustomerID',
    'Country',
    'Description',
    'Quantity',
    'UnitPrice',
    'Revenue'
]

master_df = df[master_columns]

# Sort by date for better performance
master_df = master_df.sort_values('InvoiceDate')

# ============================================
# STEP 9: DATA TYPE VERIFICATION
# ============================================

print("\n🔍 STEP 9: Verifying data types...")

print("\n   Data types in master file:")
for col in master_df.columns:
    dtype = master_df[col].dtype
    print(f"   {col:<15} : {dtype}")

# ============================================
# STEP 10: CHECK FOR ANY NULL VALUES
# ============================================

print("\n🔍 STEP 10: Checking for null values...")
null_counts = master_df.isnull().sum()
null_columns = null_counts[null_counts > 0]

if len(null_columns) > 0:
    print("   ⚠️ Warning: Null values found:")
    for col, count in null_columns.items():
        print(f"      {col}: {count} nulls")
else:
    print("   ✅ No null values found in any column!")

# ============================================
# STEP 11: SAVE TO CSV
# ============================================

print("\n💾 STEP 11: Saving to CSV...")

# Save with proper settings for Power BI
master_df.to_csv('powerbi_master_clean.csv', 
                  index=False, 
                  encoding='utf-8',
                  float_format='%.2f')  # Forces 2 decimal places

print(f"\n   ✅ File saved: powerbi_master_clean.csv")
print(f"   📊 File size: {len(master_df):,} rows × {len(master_df.columns)} columns")

# ============================================
# STEP 12: CREATE SAMPLE PREVIEW
# ============================================

print("\n📋 STEP 12: Sample preview (first 10 rows):")
print("="*70)
print(master_df.head(10).to_string())

# ============================================
# STEP 13: CREATE KPIS FOR REFERENCE
# ============================================

print("\n📊 STEP 13: Key Performance Indicators")
print("="*70)

total_revenue = master_df['Revenue'].sum()
total_orders = master_df['InvoiceNo'].nunique()
total_customers = master_df['CustomerID'].nunique()
avg_order_value = total_revenue / total_orders

print(f"💰 Total Revenue:      £{total_revenue:,.2f}")
print(f"📦 Total Orders:       {total_orders:,}")
print(f"👥 Total Customers:    {total_customers:,}")
print(f"💳 Avg Order Value:    £{avg_order_value:.2f}")
print(f"🌍 Countries:          {master_df['Country'].nunique()}")
print(f"📅 Date range:         {master_df['InvoiceDate'].min().date()} to {master_df['InvoiceDate'].max().date()}")
print(f"🏷️ Unique Products:    {master_df['Description'].nunique()}")

# ============================================
# STEP 14: SHOW TOP PRODUCTS (to verify)
# ============================================

print("\n🏆 STEP 14: Top 10 Products (verification):")
print("="*70)

top_verify = master_df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)
for i, (product, revenue) in enumerate(top_verify.items(), 1):
    print(f"   {i}. {product[:50]:<50} £{revenue:>12,.2f}")

# ============================================
# STEP 15: CREATE SUMMARY CSV (Optional)
# ============================================

print("\n💾 STEP 15: Creating summary CSV for quick reference...")

summary_df = pd.DataFrame({
    'Metric': ['Total Revenue', 'Total Orders', 'Total Customers', 'Average Order Value', 'Date Start', 'Date End', 'Unique Products'],
    'Value': [
        f'£{total_revenue:,.2f}',
        f'{total_orders:,}',
        f'{total_customers:,}',
        f'£{avg_order_value:.2f}',
        master_df['InvoiceDate'].min().date(),
        master_df['InvoiceDate'].max().date(),
        f'{master_df['Description'].nunique():,}'
    ]
})
summary_df.to_csv('powerbi_summary_reference.csv', index=False)
print("   ✅ Saved: powerbi_summary_reference.csv")

# ============================================
# FINAL VERIFICATION
# ============================================

print("\n" + "="*70)
print("✅✅✅ MASTER CSV FILE CREATED SUCCESSFULLY! ✅✅✅")
print("="*70)

print("\n📁 FILES CREATED:")
print("   1. powerbi_master_clean.csv      ← MAIN FILE (use this in Power BI)")
print("   2. powerbi_summary_reference.csv ← Reference only")

print("\n🎯 WHAT WAS REMOVED (Non-products):")
print("   ✅ POSTAGE, Manual, Bank Charges, Discount, DOTCOM POSTAGE")
print("   ✅ C2, CARRIAGE, AMAZON, BANK CHARGES")
print("   ✅ Empty descriptions")
print("   ✅ 'M' stock codes")
print("   ✅ Keywords: SHIPPING, DELIVERY, INSURANCE, GIFT WRAP, RETURNS")

print("\n🎯 OTHER CLEANING STEPS:")
print("   ✅ Removed missing CustomerID")
print("   ✅ Removed cancelled orders (C)")
print("   ✅ Removed negative quantities and prices")
print("   ✅ Removed outliers (top 1% quantity & price)")
print("   ✅ Standardized country names (EIRE → Ireland)")
print("   ✅ Removed special characters from descriptions")
print("   ✅ 2 decimal places for all currency")

print("\n📤 NEXT STEPS:")
print("   1. Copy 'powerbi_master_clean.csv' to your Windows PC")
print("   2. Open Power BI Desktop")
print("   3. Get Data → Text/CSV → Select this file")
print("   4. Click Load")
print("   5. Start building your dashboard!")

print("\n🚀 READY FOR POWER BI!")

CREATING ERROR-FREE MASTER CSV FOR POWER BI

📂 STEP 1: Loading original data...
   ✅ Loaded 541,909 rows and 8 columns

🧹 STEP 2: Cleaning data...
   Removed 135,080 rows with missing CustomerID
   Removed 8,905 cancelled orders
   Removed 0 rows with negative quantities
   Removed 40 rows with invalid prices

   ✅ Cleaned data: 397,884 rows

🚫 STEP 3: Removing non-product entries...
   Removed 1,260 rows with non-product descriptions (POSTAGE, Manual, etc.)
   Removed 0 rows with empty descriptions
   Removed 284 rows with 'M' stock codes (non-products)
   Removed rows containing non-product keywords

   ✅ After removing non-products: 391,169 rows

📊 STEP 4: Creating calculated columns...
   Removed 0 rows with invalid dates
   Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00

✏️ STEP 5: Cleaning text columns...
   Unique countries: 36

📈 STEP 6: Removing statistical outliers...
   Removed 3,820 rows with quantity > 120
   Removed 3,025 rows with price > £12.75

🔍 STEP 7: Verify